## EBITDA Monte Carlo: inputs and assumptions (corrected 24 October 2026)

**Cost definition.** The cost input is Barrick's **cost of sales per ounce sold** (US$/oz, IFRS), the variable forecast by `cost_of_sales/cost_of_sales.R`. It is *not* AISC (all-in sustaining cost: a non-GAAP measure that excludes D&A but adds sustaining capital, G&A and other items) and *not* total cash costs. The notebook uses one cost series only. Cost of sales **includes depreciation and amortisation (D&A)**, so

$$\text{EBITDA}_q = Q_q \,\big(P_q - (\text{CoS}_q - \text{D\&A}_q)\big).$$

`DA_PER_OZ` adds D&A back. With the default `DA_PER_OZ = 0` the output is *revenue minus cost of sales*, a gross-margin proxy that **understates EBITDA by D&A per ounce**. Set `DA_PER_OZ` from Barrick's MD&A (depreciation per ounce sold) to get EBITDA.

**Gold price.** GBM under the **physical (real-world) measure**. This is a forecast, not a pricing exercise, so the drift is `MU_ANNUAL` (default 0.0: no expected price appreciation, $E[S_t]=S_0$) and not the risk-free curve. Volatility comes from the GLD implied-volatility file `data/gld_chain_with_iv.csv` (2 April 2026 chain). For each expiry the notebook takes the ATM implied vol, interpolates total implied variance $\sigma^2 T$ linearly in $T$ onto the quarterly grid, and converts it to per-quarter **forward** volatilities. Before 24 October 2026 a hand-typed vol curve was used as step vols. Implied vol is a risk-neutral quantity that usually exceeds realised vol (variance risk premium), so the price fan is, if anything, conservative (wide). `gld_chain_with_iv.csv` holds only the ~100 (T, K) nodes kept by the cleaning cell (`TARGET_POINTS`), so some expiries have no strike within `ATM_BAND`. In that case the nearest strike is used, which can sit on the skew. Raise `TARGET_POINTS` in the IV notebook for a denser ATM term structure.

**Cost and production uncertainty.** Cost of sales per oz gets lognormal shocks whose horizon s.d. equals the log-space forecast s.d. exported by the R script (`Log_SD`). They are simulated as a random walk, so shocks persist like ARIMA errors. Production gets mean-preserving lognormal shocks with the relative s.d. implied by the production notebook's 80% intervals, independent across quarters as in that notebook. Price, cost and production shocks are assumed independent of each other.

**Inputs.** If `../cost_of_sales/output/cost_forecast_global.csv` and `../production/output/production_forecast_total_quarterly.csv` exist, they are used. Otherwise the notebook falls back to the arrays used in the article of 8 May 2026 and to the documented fallback s.d. values. The published cost array came from the *unweighted* cost chain, which was corrected on 24 October 2026; regenerate it with the R script.

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ==========================================
# 1. USER INPUTS & PARAMETERS
# ==========================================
rng = np.random.default_rng(42)
n_paths = 10000        # Number of Monte Carlo simulations
n_years = 5            # 5-year forecast
n_quarters = n_years * 4
dt = 0.25              # Quarterly time step; quarter i is priced at t = i * dt years after the valuation date
S0 = 4677              # Gold spot price in $/oz on the valuation date (2 April 2026)

# --- Gold-price drift: PHYSICAL measure (forecast, not pricing) ---
# Expected arithmetic annual return of gold: E[S_t] = S0 * exp(MU_ANNUAL * t).
# 0.0 = no expected appreciation (default). Alternatives: a historical mean return
# over a stated window, or expected inflation if you want a nominal drift to match
# the nominal cost path. The risk-free curve used before 24 Oct 2026 is a
# risk-neutral (pricing) drift and is not used here.
MU_ANNUAL = 0.0

# --- Cost basis: COST OF SALES per ounce sold (US$/oz), NOT AISC, NOT cash costs ---
# Cost of sales includes depreciation & amortisation; EBITDA must add it back:
#   EBITDA_q = Q_q * (P_q - (CoS_q - DA_PER_OZ_q))
# DA_PER_OZ = 0.0 -> revenue minus cost of sales (gross-margin proxy, understates
# EBITDA by D&A/oz). Scalar or length-20 array, from Barrick's MD&A.
DA_PER_OZ = 0.0

# --- Uncertainty switches ---
INCLUDE_COST_UNCERTAINTY = True
INCLUDE_PRODUCTION_UNCERTAINTY = True

# --- Input files (produced by the other components) ---
COST_FORECAST_CSV = "../cost_of_sales/output/cost_forecast_global.csv"          # cost_of_sales.R
PRODUCTION_FORECAST_CSV = "../production/output/production_forecast_total_quarterly.csv"  # production_forecast.ipynb
IV_CSV_PATH = "data/gld_chain_with_iv.csv"           # IV_gold_price_BS_inversion.ipynb
META_JSON_PATH = "data/gld_chain_wide_meta.json"     # IB fetch metadata (GLD spot S0)

ATM_BAND = 0.05        # strikes within +/-5% of the GLD spot are treated as ATM
MIN_FWD_VOL = 0.05     # floor for forward vols if total implied variance is not increasing in T

# Fallbacks = arrays used in the article of 8 May 2026 (cost array from the
# legacy UNWEIGHTED cost chain; regenerate with cost_of_sales.R).
PUBLISHED_PRODUCTION = np.array([
    704, 741, 752, 751,             #2026
    720, 743, 748, 814,             #2027
    783, 787, 789, 811,             #2028
    805, 821, 842, 814,             #2029
    802, 782, 769, 779]) * 1000     #2030   (oz per quarter)
PUBLISHED_COSTS = np.array([
    1685.07, 1698.87, 1727.07, 1783.58,     #2026
    1820.64, 1856.45, 1899.48, 1945.25,     #2027
    1986.89, 2031.28, 2077.64, 2124.60,     #2028
    2171.84, 2220.87, 2270.94, 2321.98,     #2029
    2374.12, 2427.60, 2482.20, 2538.02      #2030   (cost of sales, $/oz)
])
COST_LOG_SD_FALLBACK = 0.03   # assumption: per-quarter log s.d. of cost innovations (random walk)
PROD_REL_SD_FALLBACK = 0.05   # assumption: per-quarter relative s.d. of production

# ==========================================
# 2. COST AND PRODUCTION FORECASTS
# ==========================================
h = np.arange(1, n_quarters + 1)

if os.path.exists(COST_FORECAST_CSV):
    cost_df = pd.read_csv(COST_FORECAST_CSV)
    if len(cost_df) < n_quarters:
        raise RuntimeError(f"{COST_FORECAST_CSV} has {len(cost_df)} quarters, need {n_quarters}.")
    cost_df = cost_df.iloc[:n_quarters]
    quarterly_costs = cost_df["Global_Value"].to_numpy(float)   # median (exp of log forecast)
    cost_log_sd_h = cost_df["Log_SD"].to_numpy(float)           # forecast log s.d. at horizon h
    cost_start = pd.Period(pd.Timestamp(cost_df["Date"].iloc[0]), freq="Q")
else:
    warnings.warn(f"{COST_FORECAST_CSV} not found: using the published (legacy, unweighted) cost array "
                  f"and COST_LOG_SD_FALLBACK = {COST_LOG_SD_FALLBACK}.")
    quarterly_costs = PUBLISHED_COSTS.astype(float)
    cost_log_sd_h = COST_LOG_SD_FALLBACK * np.sqrt(h)
    cost_start = None

if os.path.exists(PRODUCTION_FORECAST_CSV):
    prod_df = pd.read_csv(PRODUCTION_FORECAST_CSV)
    if len(prod_df) < n_quarters:
        raise RuntimeError(f"{PRODUCTION_FORECAST_CSV} has {len(prod_df)} quarters, need {n_quarters}.")
    prod_df = prod_df.iloc[:n_quarters]
    quarterly_production = prod_df["Forecast_Production"].to_numpy(float)
    prod_rel_sd = prod_df["Std_Dev"].to_numpy(float) / quarterly_production
    prod_start = pd.Period(pd.Timestamp(prod_df["Date"].iloc[0]), freq="Q")
else:
    warnings.warn(f"{PRODUCTION_FORECAST_CSV} not found: using the published production array "
                  f"and PROD_REL_SD_FALLBACK = {PROD_REL_SD_FALLBACK}.")
    quarterly_production = PUBLISHED_PRODUCTION.astype(float)
    prod_rel_sd = np.full(n_quarters, PROD_REL_SD_FALLBACK)
    prod_start = None

if cost_start is not None and prod_start is not None and cost_start != prod_start:
    raise RuntimeError(f"Cost forecast starts {cost_start}, production forecast starts {prod_start}: align them.")

# ==========================================
# 3. VOLATILITY TERM STRUCTURE FROM THE IV FILE
# ==========================================
options_data = pd.read_csv(IV_CSV_PATH)
missing = [c for c in ["expiry", "T", "K", "implied_vol"] if c not in options_data.columns]
if missing:
    raise RuntimeError(f"{IV_CSV_PATH} is missing columns {missing}; run IV_gold_price_BS_inversion.ipynb first.")

with open(META_JSON_PATH, "r", encoding="utf-8") as f:
    gld_spot = float(json.load(f)["S0"])   # GLD share price: used ONLY to locate ATM strikes

options_data = options_data[np.isfinite(options_data["implied_vol"]) & (options_data["implied_vol"] > 0)]
atm_rows = []
for expiry, g in options_data.groupby("expiry"):
    moneyness = (g["K"] / gld_spot - 1.0).abs()
    near = g[moneyness <= ATM_BAND]
    if near.empty:                          # no strike in the band: nearest strike
        near = g.loc[[moneyness.idxmin()]]
    atm_rows.append({"expiry": expiry, "T": float(g["T"].median()),
                     "atm_iv": float(near["implied_vol"].median()), "n_strikes": len(near)})
iv_term = pd.DataFrame(atm_rows).sort_values("T").reset_index(drop=True)
if iv_term.empty:
    raise RuntimeError("No valid implied vols in the IV file.")

# Implied vols are TERM vols (average vol from 0 to T). Interpolate total variance
# w(T) = sigma^2 * T linearly in T; flat implied vol before the first and after the
# last expiry (the chain was fetched up to ~3 years, the forecast runs 5 years).
T_obs = iv_term["T"].to_numpy()
w_obs = iv_term["atm_iv"].to_numpy() ** 2 * T_obs
t_grid = h * dt
w_grid = np.interp(t_grid, T_obs, w_obs)
w_grid = np.where(t_grid < T_obs[0], iv_term["atm_iv"].iloc[0] ** 2 * t_grid, w_grid)
w_grid = np.where(t_grid > T_obs[-1], iv_term["atm_iv"].iloc[-1] ** 2 * t_grid, w_grid)
term_vol = np.sqrt(w_grid / t_grid)

# Per-quarter forward vols: the variance accumulated between t_{i-1} and t_i.
fwd_var = np.diff(w_grid, prepend=0.0) / dt
if np.any(fwd_var < MIN_FWD_VOL ** 2):
    warnings.warn("Total implied variance is not increasing on some quarters; forward vol floored at MIN_FWD_VOL.")
sigma_curve = np.sqrt(np.maximum(fwd_var, MIN_FWD_VOL ** 2))

print("ATM implied vol by expiry (GLD, 2 April 2026 chain):")
print(iv_term.to_string(index=False))
print("\nQuarterly grid: term vol and forward vol used in the simulation")
print(pd.DataFrame({"t_years": t_grid, "term_vol": term_vol, "forward_vol": sigma_curve}).round(4).to_string(index=False))

# ==========================================
# 4. SIMULATION: PRICE (GBM, P-measure), COST, PRODUCTION
# ==========================================
Z = rng.standard_normal((n_paths, n_quarters))
log_increments = (MU_ANNUAL - 0.5 * sigma_curve ** 2) * dt + sigma_curve * np.sqrt(dt) * Z
prices = S0 * np.exp(np.cumsum(log_increments, axis=1))   # every quarter, including Q1, is stochastic

# Cost of sales/oz: lognormal around the median forecast; random-walk log errors whose
# s.d. at horizon h equals the R forecast s.d. (persistent shocks, as in an ARIMA).
if INCLUDE_COST_UNCERTAINTY:
    cost_step_sd = np.sqrt(np.maximum(np.diff(cost_log_sd_h ** 2, prepend=0.0), 0.0))
    cost_log_err = np.cumsum(cost_step_sd * rng.standard_normal((n_paths, n_quarters)), axis=1)
    cost_paths = quarterly_costs * np.exp(cost_log_err)
else:
    cost_paths = np.broadcast_to(quarterly_costs, (n_paths, n_quarters))

# Production: mean-preserving lognormal shocks, independent across quarters.
if INCLUDE_PRODUCTION_UNCERTAINTY:
    prod_log_sd = np.sqrt(np.log1p(prod_rel_sd ** 2))
    eps = rng.standard_normal((n_paths, n_quarters))
    prod_paths = quarterly_production * np.exp(prod_log_sd * eps - 0.5 * prod_log_sd ** 2)
else:
    prod_paths = np.broadcast_to(quarterly_production, (n_paths, n_quarters))

# ==========================================
# 5. EBITDA CALCULATION & ANNUAL AGGREGATION
# ==========================================
cash_cost_per_oz = cost_paths - np.broadcast_to(DA_PER_OZ, (n_quarters,))
quarterly_ebitda_matrix = prod_paths * (prices - cash_cost_per_oz)

# Reshape (n_paths, 20) -> (n_paths, 5, 4) and sum the 4 quarters of each year.
annual_ebitda_matrix = quarterly_ebitda_matrix.reshape(n_paths, n_years, 4).sum(axis=2)

EBITDA_LABEL = ("EBITDA" if np.any(np.asarray(DA_PER_OZ) > 0)
                else "Revenue - cost of sales (EBITDA proxy, D&A not added back)")

# ==========================================
# 6. OUTPUT PERCENTILES
# ==========================================
annual_lower = np.percentile(annual_ebitda_matrix, 10, axis=0)
annual_median = np.percentile(annual_ebitda_matrix, 50, axis=0)
annual_upper = np.percentile(annual_ebitda_matrix, 90, axis=0)

annual_forecast = pd.DataFrame({
    'Year': range(1, n_years + 1),
    'Annual_EBITDA_10th_CI': annual_lower,
    'Annual_EBITDA_Median': annual_median,
    'Annual_EBITDA_90th_CI': annual_upper
})

print(f"\nForecasted annual {EBITDA_LABEL} (in Dollars):")
print(f"drift MU_ANNUAL = {MU_ANNUAL:.2%} (physical), cost uncertainty = {INCLUDE_COST_UNCERTAINTY}, "
      f"production uncertainty = {INCLUDE_PRODUCTION_UNCERTAINTY}")
print("-" * 70)
print(annual_forecast.round(0).astype(int).map(lambda x: f"{x:,}").to_string(index=False))

In [ ]:
# ==========================================
# 6. EXTRACTING COMPREHENSIVE STATISTICS
# ==========================================
# Calculate a full suite of stats across all 10,000 simulated realities
stats_df = pd.DataFrame({
    'Year': range(1, n_years + 1),
    'Mean': np.mean(annual_ebitda_matrix, axis=0),
    'Std_Dev': np.std(annual_ebitda_matrix, axis=0),
    'Min_WorstCase': np.min(annual_ebitda_matrix, axis=0),
    '10th_Pct': np.percentile(annual_ebitda_matrix, 10, axis=0),
    'Median': np.median(annual_ebitda_matrix, axis=0),
    '90th_Pct': np.percentile(annual_ebitda_matrix, 90, axis=0),
    'Max_BestCase': np.max(annual_ebitda_matrix, axis=0)
})

print("Comprehensive Annual EBITDA Statistics (in Billions):")
print("-" * 120) # Extended the dashed line to match the wider table

# Format the entire table into Billions (e.g. $8.54B) for clean reading
formatted_stats = stats_df.copy()
for col in formatted_stats.columns[1:]:
    formatted_stats[col] = formatted_stats[col].apply(lambda x: f"${x*1e-9:,.2f}B")

# The exact fix: col_space forces the width, justify aligns the headers perfectly
print(formatted_stats.to_string(index=False, col_space=14, justify='right'))

# ==========================================
# 7. PLOTTING THE INSTITUTIONAL FAN CHART
# ==========================================
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.figure(figsize=(8, 4))
years = np.arange(1, n_years + 1)

# Calculate exact percentiles to draw the shaded confidence bands
pct_5 = np.percentile(annual_ebitda_matrix, 5, axis=0)
pct_25 = np.percentile(annual_ebitda_matrix, 25, axis=0)
pct_50 = np.median(annual_ebitda_matrix, axis=0)
pct_75 = np.percentile(annual_ebitda_matrix, 75, axis=0)
pct_95 = np.percentile(annual_ebitda_matrix, 95, axis=0)

# 1. Plot the Light Blue Band (90% Confidence Interval)
plt.fill_between(years, pct_5, pct_95, color='royalblue', alpha=0.15, label='90% Confidence (5th - 95th Pct)')

# 2. Plot the Dark Blue Band (50% Confidence Interval)
plt.fill_between(years, pct_25, pct_75, color='royalblue', alpha=0.4, label='50% Core Probability (25th - 75th Pct)')

# 3. Plot the Median Line
plt.plot(years, pct_50, color='darkred', linewidth=2.5, label='Median Expected EBITDA')

# Optional: Plot just 5 "ghost" lines to show the jagged reality of a single path
plt.plot(years, annual_ebitda_matrix[:5, :].T, color='black', alpha=0.15, linewidth=1)

plt.title('Monte Carlo Simulation of Annual EBITDA (Confidence Fan Chart)')
plt.xlabel('Years')
plt.ylabel('Annual EBITDA (USD)')
plt.xticks(years, labels=['Year 1', 'Year 2', 'Year 3', 'Year 4', 'Year 5'])

# Format Y-Axis into Billions
def format_billions(x, pos):
    return f'${x*1e-9:,.1f}B'
plt.gca().yaxis.set_major_formatter(FuncFormatter(format_billions))

plt.legend(loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

In [ ]:
# ==========================================
# 7. PLOTTING THE MONTE CARLO PATHS
# ==========================================
plt.figure(figsize=(20, 10))

# Plot the first 200 ANNUAL paths
plt.plot(np.arange(1, n_years + 1), annual_ebitda_matrix[:1000, :].T, color='blue', alpha=0.05)
plt.plot(np.arange(1, n_years + 1), np.median(annual_ebitda_matrix, axis=0), color='red', linewidth=2, label='Median Annual EBITDA Path')

plt.title('Monte Carlo Simulation of Annual EBITDA')
plt.xlabel('Years')
plt.ylabel('Annual EBITDA')
plt.xticks(np.arange(1, n_years + 1), labels=['Year 1', 'Year 2', 'Year 3', 'Year 4', 'Year 5'])

# Format the Y-axis to show clean Billions of dollars (e.g., $8.0B)
def format_billions(x, pos):
    return f'${x*1e-9:,.1f}B'
plt.gca().yaxis.set_major_formatter(FuncFormatter(format_billions))

plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()